# SpaLiFT, step by step

Run this notebook from top to bottom with **Shift+Enter**. You will create tiny artificial inputs, train a model on CPU, and inspect three exports: an embedding, a generated molecular profile, and a correspondence matrix. Basic Python knowledge is enough.

The synthetic example uses two slices, two modalities, 24 spots per context, and 32 genes.

## Before you start

Keep a local clone of the **SpaLiFT source repository** and save this notebook inside it before launching Jupyter. If you keep the notebook in the sibling tutorial repository instead, launch Jupyter from their common parent directory so both repositories are visible. The path cell below usually finds the source automatically.

Use Python **3.10–3.12**. Install SpaLiFT and the notebook tools into the environment you want to use, then select that environment as this notebook's kernel. For example, run these commands in a terminal **from the SpaLiFT source repository**:

```sh
python -m pip install -e .
python -m pip install ipykernel jupyterlab
python -m ipykernel install --user --name spalift --display-name "Python (SpaLiFT)"
python -m jupyterlab
```

If your environment already has the package and a registered kernel, use it. The notebook does not install packages during **Run All**; its kernel must already contain SpaLiFT's dependencies. JupyterLab is optional if you use another notebook editor.

Saved outputs below come from one completed synthetic CPU run. Use **Restart Kernel and Run All** to create your own fresh results.

The steps are: **check the environment → locate the source → make and inspect data → configure and train → export and inspect results**.


## 1. Check the Python kernel

Confirm that the selected kernel uses a supported Python version. If an import fails later, install the missing dependency in this kernel's environment and restart the kernel.


In [1]:
import sys
from pathlib import Path

assert (3, 10) <= sys.version_info[:2] < (3, 13), "Select a Python 3.10–3.12 kernel."
print("Python:", sys.version.split()[0])
print("Interpreter:", Path(sys.executable).name)


Python: 3.10.16
Interpreter: python.exe


## 2. Locate the SpaLiFT source

Leave `SOURCE_REPO = None` to search the working directory, its parents, and nearby `SpaLiFT` folders. If needed, set it to your local source folder, for example `Path.home() / "projects" / "SpaLiFT"`. This must be the source repository containing `src/spalift`, not the tutorial website.


In [2]:
SOURCE_REPO = None  # Or your local source path: Path.home() / "projects" / "SpaLiFT"

if SOURCE_REPO is not None:
    candidates = [Path(SOURCE_REPO).expanduser()]
else:
    candidates = [
        candidate
        for base in (Path.cwd(), *Path.cwd().parents)
        for candidate in (base, base / "SpaLiFT", base / "code" / "SpaLiFT")
    ]

required_files = ("src/spalift/config.py", "configs/demo.yaml", "examples/make_demo_data.py")
source_repo = next(
    (candidate.resolve() for candidate in candidates
     if all((candidate / name).is_file() for name in required_files)),
    None,
)
if source_repo is None:
    raise FileNotFoundError("Set SOURCE_REPO to your SpaLiFT source repository, then rerun this cell.")
sys.path.insert(0, str(source_repo / "src"))
print("Source repository:", source_repo.name)


Source repository: SpaLiFT


## 3. Import the packages

Import PyTorch before AnnData to support the tested Windows runtime. Adding `src` above selects the local SpaLiFT code; its third-party dependencies still need to be installed in the kernel.


In [3]:
import torch
import anndata as ad
import h5py
import numpy as np
import pandas as pd
import scipy.sparse as sparse
import yaml
import subprocess
import tempfile
from uuid import uuid4

from spalift.config import RunConfig
from spalift.exports import export
from spalift.trainer import train

print(f"PyTorch {torch.__version__} | AnnData {ad.__version__} | NumPy {np.__version__}")


PyTorch 2.9.1+cu126 | AnnData 0.11.3 | NumPy 1.25.2


## 4. Create a fresh workspace

Each execution of this cell creates a new folder under the source repository's `outputs/notebook_runs`. Data, model runs, and exports live in separate subfolders. Running the whole notebook again preserves previous runs.

After rerunning this cell, rerun the following cells in order so they use the new workspace.


In [4]:
runs_root = source_repo / "outputs" / "notebook_runs"
runs_root.mkdir(parents=True, exist_ok=True)
workspace = Path(tempfile.mkdtemp(prefix="walkthrough_", dir=runs_root))
data_dir = workspace / "data"
export_dir = workspace / "exports"
export_dir.mkdir()
print("Workspace:", workspace.relative_to(source_repo).as_posix())


Workspace: outputs/notebook_runs/walkthrough_xaxp1k2f


## 5. Generate the artificial inputs

Use the source repository's generator with the **current kernel's Python interpreter**. It creates four H5AD files in the new data folder. To regenerate the inputs, rerun the workspace cell first; the generator refuses to overwrite existing data.


In [5]:
generator_result = subprocess.run(
    [sys.executable, str(source_repo / "examples" / "make_demo_data.py"),
     "--output-dir", str(data_dir)],
    check=True, capture_output=True, text=True,
)
input_paths = sorted(data_dir.glob("*.h5ad"))
assert len(input_paths) == 4
print("Created", len(input_paths), "files:", ", ".join(path.name for path in input_paths))


Created 4 files: a_atac.h5ad, a_rna.h5ad, b_atac.h5ad, b_rna.h5ad


## 6. Inspect the inputs

One **context** means one slice and one modality. `X` holds non-negative, linear-scale RNA values or ATAC-derived **gene scores**, not ATAC peaks. The loader will perform normalization, `log1p`, and training-fitted standardization.


In [6]:
datasets = {path.stem: ad.read_h5ad(path) for path in input_paths}
input_summary = []
for name, data in datasets.items():
    slice_id, modality = name.split("_")
    available = (int(data.var["genescore_annotation_available"].sum())
                 if modality == "atac" else data.n_vars)
    input_summary.append({
        "slice": slice_id, "modality": modality,
        "spots": data.n_obs, "genes": data.n_vars,
        "spatial shape": str(data.obsm["spatial"].shape),
        "available genes": available,
    })
pd.DataFrame(input_summary)


,slice,modality,spots,genes,spatial shape,available genes
0,a,atac,24,32,"(24, 2)",32
1,a,rna,24,32,"(24, 2)",32
2,b,atac,24,32,"(24, 2)",32
3,b,rna,24,32,"(24, 2)",32


Check the identities and values before training. All contexts need the same ordered gene axis. Paired modalities within a slice need matching spot sets and coordinates; the generator already writes them in matching row order.


In [7]:
source = datasets["a_rna"]
target = datasets["b_atac"]
for data in datasets.values():
    values = data.X.data if sparse.issparse(data.X) else np.asarray(data.X)
    assert np.isfinite(values).all() and (values >= 0).all()
    assert data.var_names.equals(source.var_names)
    assert data.obs_names.is_unique and data.var_names.is_unique
for slice_id in ("a", "b"):
    rna, atac = datasets[f"{slice_id}_rna"], datasets[f"{slice_id}_atac"]
    assert rna.obs_names.equals(atac.obs_names)
    np.testing.assert_allclose(rna.obsm["spatial"], atac.obsm["spatial"], atol=1e-6)
print("Input checks passed.")
print("Source spots:", source.obs_names[:3].tolist())
print("Genes:", source.var_names[:3].tolist())


Input checks passed.
Source spots: ['a_spot_00', 'a_spot_01', 'a_spot_02']
Genes: ['gene_00', 'gene_01', 'gene_02']


## 7. Configure a short CPU run

Load the supplied demo configuration, then point every context to our newly generated files using absolute paths. The demo uses a small network and five updates per epoch. Two epochs are enough to test the workflow, not to assess scientific performance.


In [8]:
config = RunConfig.load(source_repo / "configs" / "demo.yaml")
for context in config.contexts:
    context["path"] = str(data_dir / f"{context['slice']}_{context['modality']}.h5ad")
config.output_dir = str(workspace / "training" / f"run_{uuid4().hex}")
config.device = "cpu"
config.epochs = 2
config.steps_per_epoch = 5
config.validate()
print({
    "device": config.device, "epochs": config.epochs,
    "steps_per_epoch": config.steps_per_epoch, "latent_dim": config.latent_dim,
    "contexts": len(config.contexts),
})


{'device': 'cpu', 'epochs': 2, 'steps_per_epoch': 5, 'latent_dim': 8, 'contexts': 4}


## 8. Train

This cell calls the Python API. Each execution selects a new training directory and saves a resolved YAML configuration, so you can rerun training without overwriting a model. Training prints one progress line per epoch and selects its checkpoint using the validation objective.


In [9]:
run_id = uuid4().hex
run_dir = workspace / "training" / f"run_{run_id}"
run_config = RunConfig.from_dict({**config.to_dict(), "output_dir": str(run_dir)})
config_path = workspace / f"config_{run_id}.yaml"
with config_path.open("x", encoding="utf-8") as handle:
    yaml.safe_dump(run_config.to_dict(), handle, sort_keys=False)

summary = train(run_config)
checkpoint = run_dir / "checkpoint.pt"
assert summary["status"] == "complete" and checkpoint.is_file()
print("Checkpoint:", checkpoint.relative_to(workspace).as_posix())


epoch 1/2 validation_objective=1.066843 elapsed=0.2s


epoch 2/2 validation_objective=1.041589 elapsed=0.3s


Checkpoint: training/run_60ad66be882346589fa6a3e9cb2c11a3/checkpoint.pt


Inspect the training summary. The validation objective is a training diagnostic; it is not a biological accuracy score. The complete history and configuration are saved alongside the checkpoint.


In [10]:
print({
    "status": summary["status"], "updates": summary["total_updates"],
    "best_epoch": summary["best_epoch"],
    "best_validation_loss": round(summary["best_validation_loss"], 6),
})
pd.DataFrame(summary["history"])[["epoch", "validation_loss", "updates"]]


{'status': 'complete', 'updates': 10, 'best_epoch': 2, 'best_validation_loss': 1.041589}


,epoch,validation_loss,updates
0,1,1.066843,5
1,2,1.041589,5


## 9. Encode the source RNA spots

Export the source slice `a` RNA embedding. Exports are standard **HDF5 `.h5` files**, read with `h5py`; they are not AnnData `.h5ad` files. Each export cell chooses a fresh filename, so it can be rerun independently after training.


In [11]:
embedding_path = export_dir / f"a_rna_embedding_{uuid4().hex}.h5"
export(
    checkpoint, source_path=data_dir / "a_rna.h5ad", source_modality="rna",
    output_path=embedding_path, kind="encode", device="cpu",
)
print("Saved:", embedding_path.name)


Saved: a_rna_embedding_a25de1010dbb4c488e31a0fbc5663d32.h5


The embedding has one row per source spot and `latent_dim` columns. Check its source identifiers and gene metadata against the input.


In [12]:
with h5py.File(embedding_path, "r") as result:
    assert result.attrs["status"] == "complete"
    z = result["z"][:]
    embedding_spots = result["source_spots"].asstr()[:]
    embedding_genes = result["genes"].asstr()[:]
assert z.shape == (source.n_obs, run_config.latent_dim)
assert np.isfinite(z).all()
np.testing.assert_array_equal(embedding_spots, source.obs_names.to_numpy())
np.testing.assert_array_equal(embedding_genes, source.var_names.to_numpy())
print("Embedding shape:", z.shape)
print("First source IDs:", embedding_spots[:3].tolist())


Embedding shape: (24, 8)
First source IDs: ['a_spot_00', 'a_spot_01', 'a_spot_02']


## 10. Generate ATAC gene scores conditioned on slice b

Request the observed target modality `atac` and target slice `b`. The output still has one row per **source slice a spot**, at the source coordinates. This changes the molecular target context; it does not place predictions on slice b's physical grid. Entirely unseen target slices or untrained modalities are unsupported.


In [13]:
generated_path = export_dir / f"a_to_b_atac_{uuid4().hex}.h5"
torch.manual_seed(run_config.seed)  # Repeatable sampling in the same environment.
export(
    checkpoint, source_path=data_dir / "a_rna.h5ad", source_modality="rna",
    output_path=generated_path, kind="generate",
    target_modality="atac", target_slice="b", device="cpu",
)
print("Saved:", generated_path.name)


Saved: a_to_b_atac_ef077344c5714ef7bc5136dc9849c831.h5


Inspect both output scales. `standardized` contains model-space values; `log_normalized` applies the saved target preprocessor's inverse standardization. The latter remains continuous and unclipped, may contain negative values, and is **not raw counts**. The availability mask identifies target genes supported by the input annotation.


In [14]:
with h5py.File(generated_path, "r") as result:
    assert result.attrs["status"] == "complete"
    standardized = result["standardized"][:]
    log_normalized = result["log_normalized"][:]
    available = result["target_feature_available"][:]
    np.testing.assert_array_equal(result["source_spots"].asstr()[:], source.obs_names.to_numpy())
    np.testing.assert_array_equal(result["genes"].asstr()[:], source.var_names.to_numpy())
    np.testing.assert_allclose(result["spatial"][:], source.obsm["spatial"])
assert standardized.shape == log_normalized.shape == (source.n_obs, source.n_vars)
assert available.shape == (source.n_vars,)
assert np.isfinite(standardized).all() and np.isfinite(log_normalized).all()
print("Generated shape:", log_normalized.shape, "| Available genes:", int(available.sum()))
pd.DataFrame([
    {"scale": name, "minimum": float(values.min()), "maximum": float(values.max())}
    for name, values in (("standardized", standardized), ("log_normalized", log_normalized))
])


Generated shape: (24, 32) | Available genes: 32


,scale,minimum,maximum
0,standardized,-21.790924,24.705069
1,log_normalized,-35.888302,65.290268


## 11. Export full correspondence

Supply an actual target H5AD file to obtain `P`, a source-by-target matrix. Here the rows are slice a RNA spots and the columns are slice b ATAC spots. Values are learned correspondence weights, not verified biological matches. Full exports require storage proportional to the number of source spots times the number of target spots.


In [15]:
correspondence_path = export_dir / f"a_to_b_correspondence_{uuid4().hex}.h5"
export(
    checkpoint, source_path=data_dir / "a_rna.h5ad", source_modality="rna",
    output_path=correspondence_path, kind="correspondence",
    target_path=data_dir / "b_atac.h5ad", target_modality="atac",
    target_slice="b", device="cpu",
)
print("Saved:", correspondence_path.name)


Saved: a_to_b_correspondence_468c0232d4ab44a98052424a0a610c71.h5


Check the row and column identifiers, matrix shape, and normalization. Each source row sums to approximately one over the complete target spot set.


In [16]:
with h5py.File(correspondence_path, "r") as result:
    assert result.attrs["status"] == "complete"
    P = result["P"][:]
    source_spots = result["source_spots"].asstr()[:]
    target_spots = result["target_spots"].asstr()[:]
assert P.shape == (source.n_obs, target.n_obs)
assert np.isfinite(P).all() and (P >= 0).all()
np.testing.assert_array_equal(source_spots, source.obs_names.to_numpy())
np.testing.assert_array_equal(target_spots, target.obs_names.to_numpy())
row_sums = P.sum(axis=1)
np.testing.assert_allclose(row_sums, 1.0, atol=1e-6)
print("Correspondence shape:", P.shape)
print(f"Row-sum range: {row_sums.min():.6f} to {row_sums.max():.6f}")
print("First target IDs:", target_spots[:3].tolist())


Correspondence shape: (24, 24)
Row-sum range: 1.000000 to 1.000000
First target IDs: ['b_spot_00', 'b_spot_01', 'b_spot_02']
